# AxonScope Colab CPU/GPU Hotpaths

Use this notebook in a Google Colab GPU runtime. It clones the moving `bench-colab` branch, installs AxonScope, runs matching warm hotpath scale probes on the GPU backend and on a forced CPU backend, stores both results under one `benchmark/results/hotpaths/` folder, writes a small comparison CSV, zips the run folder, and downloads it directly through the browser.

Local setup before running this notebook:

```bash
git add -A
git commit -m "Benchmark Colab run"
make bench-colab-push
```

After download, unzip the archive into your local `benchmark/results/hotpaths/` folder.

In [ ]:
import csv
import datetime
import json
import os
import pathlib
import shutil
import subprocess

from google.colab import files

# Replace this once with the real repository URL.
REPO_URL = "https://github.com/louisreg/AxonScope.git"
BRANCH = "bench-colab"
PKG_DIR = pathlib.Path("/content/AxonScope")

WORKLOAD = "all"
PRESET = "scale"
WARMUPS = 1
SWEEP_REPEATS = 3
RUN_GPU = True
RUN_CPU = True

run_id = datetime.datetime.now().strftime("colab_cpu_gpu_%Y%m%d_%H%M%S")


def sh(command, cwd=None, env=None):
    print(f"\n$ {command}")
    subprocess.run(command, shell=True, cwd=cwd, check=True, env=env)


def verify_gpu_backend():
    sh(
        "python - <<'PY'\n"
        "import jax\n"
        "backend = jax.default_backend()\n"
        "print('jax backend:', backend)\n"
        "print('jax devices:', jax.devices())\n"
        "if backend != 'gpu':\n"
        "    raise SystemExit('Colab runtime is not using a GPU backend.')\n"
        "PY",
        cwd=PKG_DIR,
    )


def verify_cpu_backend(cpu_env):
    sh(
        "python - <<'PY'\n"
        "import jax\n"
        "backend = jax.default_backend()\n"
        "print('forced jax backend:', backend)\n"
        "print('jax devices:', jax.devices())\n"
        "if backend != 'cpu':\n"
        "    raise SystemExit('Forced CPU backend did not activate.')\n"
        "PY",
        cwd=PKG_DIR,
        env=cpu_env,
    )


def run_hotpaths(label, *, env=None):
    sh(
        "python benchmark/hotpaths/run.py "
        f"--workload {WORKLOAD} "
        f"--preset {PRESET} "
        f"--warmups {WARMUPS} "
        f"--sweep-repeats {SWEEP_REPEATS} "
        f"--prefix {label} "
        f"--out-dir benchmark/results/hotpaths/{run_id} "
        "--no-print-summary",
        cwd=PKG_DIR,
        env=env,
    )


def load_manifest(label):
    path = results_root / run_id / label / "manifest.json"
    return json.loads(path.read_text())


STAGES = [
    "simulation.pool.total",
    "dispatch.build_plan",
    "runtime.prepare",
    "inputs.intracellular",
    "inputs.extracellular",
    "kernel.enqueue",
    "kernel.wait",
    "results.split_batch",
    "results.to_public",
]


def stage_totals(run):
    return {row["name"]: float(row["total_ms"]) for row in run.get("summary", [])}


def runs_by_key(manifest):
    return {(run["workload"], int(run["size"])): run for run in manifest["runs"]}


def write_comparison_summary(labels):
    if not {"gpu", "cpu"}.issubset(labels):
        return None

    gpu_runs = runs_by_key(load_manifest("gpu"))
    cpu_runs = runs_by_key(load_manifest("cpu"))
    keys = sorted(gpu_runs.keys() & cpu_runs.keys(), key=lambda item: (item[0], item[1]))

    columns = ["workload", "size", "gpu_total_ms", "cpu_total_ms", "cpu_over_gpu_total"]
    for stage in STAGES:
        safe = stage.replace(".", "_")
        columns.extend([f"gpu_{safe}_ms", f"cpu_{safe}_ms", f"cpu_over_gpu_{safe}"])

    rows = []
    for workload, size in keys:
        gpu_stage = stage_totals(gpu_runs[(workload, size)])
        cpu_stage = stage_totals(cpu_runs[(workload, size)])
        gpu_total = gpu_stage.get("simulation.pool.total", 0.0)
        cpu_total = cpu_stage.get("simulation.pool.total", 0.0)
        row = {
            "workload": workload,
            "size": size,
            "gpu_total_ms": gpu_total,
            "cpu_total_ms": cpu_total,
            "cpu_over_gpu_total": (cpu_total / gpu_total) if gpu_total else "",
        }
        for stage in STAGES:
            safe = stage.replace(".", "_")
            gpu_value = gpu_stage.get(stage, 0.0)
            cpu_value = cpu_stage.get(stage, 0.0)
            row[f"gpu_{safe}_ms"] = gpu_value
            row[f"cpu_{safe}_ms"] = cpu_value
            row[f"cpu_over_gpu_{safe}"] = (cpu_value / gpu_value) if gpu_value else ""
        rows.append(row)

    summary_path = results_root / run_id / "comparison_summary.csv"
    with summary_path.open("w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)

    print("\nCPU/GPU comparison summary:")
    for row in rows:
        print(
            f"{row['workload']} n{row['size']}: "
            f"GPU {row['gpu_total_ms']:.1f} ms | "
            f"CPU {row['cpu_total_ms']:.1f} ms | "
            f"CPU/GPU {row['cpu_over_gpu_total']:.2f}x"
        )
    return summary_path


sh(f"rm -rf {PKG_DIR}")
sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {PKG_DIR}")
sh("git rev-parse --short HEAD", cwd=PKG_DIR)

sh("python -m pip install -U pip", cwd=PKG_DIR)
sh('python -m pip install -e ".[examples,benchmark]"', cwd=PKG_DIR)
sh("nvidia-smi || true", cwd=PKG_DIR)

cpu_env = {**os.environ, "JAX_PLATFORMS": "cpu"}
verify_gpu_backend()
verify_cpu_backend(cpu_env)

results_root = PKG_DIR / "benchmark/results/hotpaths"
(results_root / run_id).mkdir(parents=True, exist_ok=True)

completed_labels = []
if RUN_GPU:
    run_hotpaths("gpu")
    completed_labels.append("gpu")
if RUN_CPU:
    run_hotpaths("cpu", env=cpu_env)
    completed_labels.append("cpu")

summary_path = write_comparison_summary(set(completed_labels))
run_dir = results_root / run_id
zip_path = shutil.make_archive(
    str(results_root / run_id),
    "zip",
    root_dir=results_root,
    base_dir=run_id,
)
print(f"\nResults folder: {run_dir}")
if summary_path is not None:
    print(f"Comparison CSV: {summary_path}")
print(f"Archive: {zip_path}")

try:
    ipython = get_ipython()  # Available in notebook cells.
except NameError:
    ipython = None

if ipython is not None and getattr(ipython, "kernel", None) is not None:
    files.download(zip_path)
else:
    print("\nDownload is available only from a Colab notebook cell.")
    print("Run this in a notebook cell:")
    print("from google.colab import files")
    print(f"files.download({zip_path!r})")